In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q datasets pandas sentence-transformers faiss-cpu transformers accelerate bitsandbytes rank_bm25 word2number

import torch
import gc
import logging

print("Dependencies installed.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.8 MB/s eta 0:00:00:00:0100:01
Dependencies installed.


In [2]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import faiss

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("/kaggle/working/legal_triage.log"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("legal_triage")

# Walk input dir (info only)
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        logger.debug(os.path.join(dirname, filename))

In [3]:
QA_DIR = "/kaggle/input/datasets/kashanali446/qa-faiss-store/qa_faiss_store"


def _normalize_index_in_memory(index, name="index"):
    n, d = index.ntotal, index.d
    vecs = index.reconstruct_n(0, n).astype("float32")
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    safe = np.clip(norms, 1e-8, None)
    vecs = vecs / safe
    new_index = faiss.IndexFlatIP(d)
    new_index.add(vecs)
    logger.info(
        f"{name}: {n} vectors (dim={d}) L2-normalized -> IndexFlatIP "
        f"norm range before: {norms.min():.3f}..{norms.max():.3f}"
    )
    del vecs
    return new_index


logger.info("Loading prebuilt Q&A index...")
qa_index_raw = faiss.read_index(f"{QA_DIR}/qa_index.faiss")
with open(f"{QA_DIR}/qa_documents.pkl", "rb") as f:
    rag_documents = pickle.load(f)

assert qa_index_raw.ntotal == len(rag_documents), (
    f"Mismatch: index has {qa_index_raw.ntotal} vectors but "
    f"{len(rag_documents)} documents were loaded. Store may be corrupted."
)

qa_index = _normalize_index_in_memory(qa_index_raw, "qa_index")
logger.info(
    f"Q&A index ready: {qa_index.ntotal} vectors, {len(rag_documents)} documents"
)

2026-09-22 13:52:02,110 [INFO] Loading prebuilt Q&A index...
2026-09-22 13:52:02,386 [INFO] qa_index: 3742 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:02,387 [INFO] Q&A index ready: 3742 vectors, 3742 documents


In [4]:
STATUTES_DIR = "/kaggle/input/datasets/kashanali446/statutes-faiss-store/statutes_faiss_store"

logger.info("Loading prebuilt per-state statute indices...")
with open(f"{STATUTES_DIR}/manifest.json") as f:
    manifest = json.load(f)

state_indices = {}
state_docs_map = {}
state_bm25 = {}

for code in manifest:
    raw = faiss.read_index(f"{STATUTES_DIR}/state_{code}_index.faiss")
    state_indices[code] = _normalize_index_in_memory(raw, f"state_{code}")
    with open(f"{STATUTES_DIR}/state_{code}_docs.pkl", "rb") as f:
        state_docs_map[code] = pickle.load(f)
    with open(f"{STATUTES_DIR}/state_{code}_bm25.pkl", "rb") as f:
        state_bm25[code] = pickle.load(f)

for code in state_indices:
    n_index = state_indices[code].ntotal
    n_docs = len(state_docs_map[code])
    assert n_index == n_docs, (
        f"Mismatch for '{code}': index has {n_index} vectors but "
        f"{n_docs} documents."
    )

logger.info(f"Loaded {len(state_indices)} state indices: {sorted(state_indices.keys())}")

2026-09-22 13:52:10,991 [INFO] Loading prebuilt per-state statute indices...
2026-09-22 13:52:11,700 [INFO] state_ak: 17035 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:14,252 [INFO] state_al: 41895 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:17,281 [INFO] state_ar: 34673 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:19,823 [INFO] state_az: 22422 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:23,396 [INFO] state_ca: 161525 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:29,531 [INFO] state_co: 28172 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:31,452 [INFO] state_ct: 15765 vectors (dim=384) L2-normalized -> IndexFlatIP norm range before: 1.000..1.000
2026-09-22 13:52:33,105 [INFO] state_dc:

In [5]:
from sentence_transformers import SentenceTransformer

logger.info("Loading embedding model on CPU...")
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
logger.info("Embedding model ready.")

free, total = torch.cuda.mem_get_info()
logger.info(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

2026-09-22 13:54:39,421 [INFO] TensorFlow version 2.20.0 available.
2026-09-22 13:54:39,425 [INFO] JAX version 0.7.2 available.
2026-09-22 13:54:46,199 [INFO] Loading embedding model on CPU...
2026-09-22 13:54:46,374 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:54:46,375 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-22 13:54:46,388 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-09-22 13:54:46,402 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-09-22 13:54:46,494 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:54:46,506 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-22 13:54:46,522 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

2026-09-22 13:54:46,535 [INFO] Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-09-22 13:54:46,615 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:54:46,629 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-09-22 13:54:46,707 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:54:46,718 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"
2026-09-22 13:54:46,733 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "

...

2026-09-22 13:54:51,523 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-09-22 13:54:51,609 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:54:51,622 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/special_tokens_map.json "HTTP/1.1 200 OK"
2026-09-22 13:54:51,637 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

2026-09-22 13:54:51,729 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-09-22 13:54:51,951 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:54:51,965 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-09-22 13:54:51,979 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-09-22 13:54:52,080 [INFO] HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-09-22 13:54:52,091 [INFO] Embedding model ready.
2026-09-22 13:54:52,436 [INFO] GPU free: 15.53 GB / 15.64 GB


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "NousResearch/Hermes-3-Llama-3.1-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

logger.info(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

logger.info("Loading LLM (4-bit quantized)...")
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
logger.info("LLM ready.")

2026-09-22 13:56:33,081 [INFO] Loading tokenizer for NousResearch/Hermes-3-Llama-3.1-8B...
2026-09-22 13:56:33,215 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:56:33,228 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/config.json "HTTP/1.1 200 OK"
2026-09-22 13:56:33,242 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

2026-09-22 13:56:33,339 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:56:33,352 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/tokenizer_config.json "HTTP/1.1 200 OK"
2026-09-22 13:56:33,366 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-09-22 13:56:33,463 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:56:33,475 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/tokenizer_config.json "HTTP/1.1 200 OK"
2026-09-22 13:56:33,563 [INFO] HTTP Request: GET https://huggingface.co/api/models/NousResearch/Hermes-3-Llama-3.1-8B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-09-22 13:56:33,647 [INFO] HTTP Request: GET https://huggingface.co/api/models/NousResearch/Hermes-3-Llama-3.1-8B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-09-22 13:56:33,727 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:56:33,738 [INFO] HTTP Request: HEAD https:/

...

2026-09-22 13:56:40,417 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/896ea440e5a9e6070e3d8a2774daf2b481ab425b/model-00001-of-00004.safetensors "HTTP/1.1 302 Found"
2026-09-22 13:56:40,444 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/896ea440e5a9e6070e3d8a2774daf2b481ab425b/model-00003-of-00004.safetensors "HTTP/1.1 302 Found"
2026-09-22 13:56:40,465 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/896ea440e5a9e6070e3d8a2774daf2b481ab425b/model-00002-of-00004.safetensors "HTTP/1.1 302 Found"
2026-09-22 13:56:40,473 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/896ea440e5a9e6070e3d8a2774daf2b481ab425b/model-00004-of-00004.safetensors "HTTP/1.1 302 Found"
2026-09-22 13:56:40,503 [INFO] HTTP Request: GET https://huggingface.co/api/models/NousResearch/Hermes-3-Llama-3.1-8B/xet-read-token/896ea440e5a9e6070e3d8a2

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-09-22 13:58:53,592 [INFO] HTTP Request: HEAD https://huggingface.co/NousResearch/Hermes-3-Llama-3.1-8B/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-22 13:58:53,608 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/generation_config.json "HTTP/1.1 200 OK"
2026-09-22 13:58:53,624 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/NousResearch/Hermes-3-Llama-3.1-8B/896ea440e5a9e6070e3d8a2774daf2b481ab425b/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

2026-09-22 13:58:53,993 [INFO] LLM ready.


In [16]:
from word2number import w2n

# ---------------------------------------------------------------------------
# STATE NORMALIZATION
# ---------------------------------------------------------------------------

_STATE_NAME_TO_CODE = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar",
    "california": "ca", "colorado": "co", "connecticut": "ct", "delaware": "de",
    "district of columbia": "dc", "washington dc": "dc", "washington d.c.": "dc",
    "florida": "fl", "georgia": "ga", "hawaii": "hi", "idaho": "id",
    "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks",
    "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms",
    "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv",
    "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm", "new york": "ny",
    "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok",
    "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut",
    "vermont": "vt", "virginia": "va", "washington": "wa", "west virginia": "wv",
    "wisconsin": "wi", "wyoming": "wy", "puerto rico": "pr",
    "federal": "federal", "united states": "federal", "us federal": "federal",
}


def normalize_state(raw, valid_codes):
    """Return a lowercase 2-letter code (or 'federal') present in valid_codes, else None."""
    if raw is None:
        return None
    s = str(raw).strip().lower()
    if not s:
        return None
    # strip trailing punctuation
    s = re.sub(r"[^\w\s.]", "", s).strip()
    if s in valid_codes:
        return s
    if s in _STATE_NAME_TO_CODE:
        code = _STATE_NAME_TO_CODE[s]
        return code if code in valid_codes else None
    # try partial match for "state of texas" etc.
    for name, code in _STATE_NAME_TO_CODE.items():
        if name in s and code in valid_codes:
            return code
    return None


# ---------------------------------------------------------------------------
# AMOUNT NORMALIZATION
# ---------------------------------------------------------------------------

_MULTIPLIERS = {"k": 1_000, "m": 1_000_000, "b": 1_000_000_000}


def parse_amount(raw):
    """Parse an amount from many representations. Returns positive float or None."""
    if raw is None:
        return None
    if isinstance(raw, (int, float)):
        return float(raw) if raw > 0 else None

    s = str(raw).strip()
    if not s:
        return None

    s_lower = s.lower()
    # strip currency markers
    for token in ("usd", "$", "dollars", "dollar", "bucks", "buck"):
        s_lower = s_lower.replace(token, " ")
    s_lower = s_lower.replace(",", " ").strip()
    s_lower = re.sub(r"\s+", " ", s_lower)

    # handle "1k", "1.5m"
    m = re.match(r"^(\d+(?:\.\d+)?)\s*([kmb])$", s_lower)
    if m:
        return float(m.group(1)) * _MULTIPLIERS[m.group(2)]

    # plain number
    try:
        v = float(s_lower)
        return v if v > 0 else None
    except ValueError:
        pass

    # spelled out
    try:
        v = float(w2n.word_to_num(s_lower))
        return v if v > 0 else None
    except Exception:
        pass

    return None

In [18]:
_ALLOWED_KEYS = [
    "case_type", "state", "disputed_amount",
    "written_contract_exists", "payment_status",
]

_VALID_PAYMENT_STATUS = {"unpaid", "partially_paid", "paid", "disputed"}


def _normalize_null(v):
    if isinstance(v, str) and v.strip().lower() in ("null", "none", "unknown", ""):
        return None
    return v


def _validate_delta(delta, state_indices):
    """Whitelist and normalize every field. Drop anything invalid."""
    if not isinstance(delta, dict):
        return {}
    cleaned = {}

    # case_type
    ct = _normalize_null(delta.get("case_type"))
    if isinstance(ct, str) and ct.strip():
        cleaned["case_type"] = ct.strip().lower()

    # state (accept full names, codes, "federal")
    st = _normalize_null(delta.get("state"))
    if isinstance(st, str):
        code = normalize_state(st, state_indices)
        if code:
            cleaned["state"] = code

    # disputed_amount
    amt = _normalize_null(delta.get("disputed_amount"))
    parsed = parse_amount(amt)
    if parsed is not None:
        cleaned["disputed_amount"] = parsed

    # written_contract_exists
    wc = delta.get("written_contract_exists")
    if isinstance(wc, bool):
        cleaned["written_contract_exists"] = wc
    elif isinstance(wc, str) and wc.strip().lower() in ("true", "false"):
        cleaned["written_contract_exists"] = wc.strip().lower() == "true"

    # payment_status
    ps = _normalize_null(delta.get("payment_status"))
    if isinstance(ps, str) and ps.strip().lower() in _VALID_PAYMENT_STATUS:
        cleaned["payment_status"] = ps.strip().lower()

    return cleaned

In [19]:
def translate_query(raw_user_input, last_assistant_message, previous_query=None):
    """Rewrite the user's raw input into a standalone legal search query."""
    system_prompt = """You are a Query Translation Engine for a legal database.
Rewrite the user's raw input into a clear, professional, standalone legal search query.

If a "Previous Legal Query" is provided and the user is adding a new fact
(not asking a new question), rewrite a single combined query that incorporates
both the previous query and the new fact.

If the user is answering a clarification question, use the Previous Assistant
Message only to resolve references like "yes", "no", "that one".

Respond ONLY with the rewritten query. No quotes, no explanations, no markdown.
Do not invent a jurisdiction, statute number, or party name the user did not mention.
Do NOT copy action items, procedural suggestions, or statute names out of the
Previous Assistant Message into the rewritten query."""

    user_block = f"Previous Assistant Message: {last_assistant_message}\n\n"
    if previous_query:
        user_block += f"Previous Legal Query: {previous_query}\n\n"
    user_block += f"Raw User Input: {raw_user_input}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_block},
    ]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(
        **inputs, max_new_tokens=140,
        do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def update_intake_state(user_query, current_state, state_indices, last_assistant_message="None"):
    system_prompt = f"""You are a strict fact-extraction tool for a legal intake system.

Read ONLY the user's latest message and emit a JSON object containing ONLY the
fields the user has EXPLICITLY stated in that message.

ABSOLUTE RULES — violating any of these is a failure:
1. NEVER guess, infer, assume, or fill in a value the user did not state.
2. NEVER echo back a value from a previous turn. Output a DELTA, not a full state.
3. If the message contains no new extractable facts, output exactly: {{}}
4. Only these keys may appear: {_ALLOWED_KEYS}
5. "state" → a US state name OR two-letter postal code. Emit ONLY if the user
   explicitly names a state. Do NOT infer a state from a city, an area code,
   a court name, a time zone, or the topic of the question.
6. "written_contract_exists" → true/false. Emit ONLY if the user explicitly
   says there is or is not a WRITTEN contract.
7. "disputed_amount" → a number in USD. Emit ONLY if a specific amount is
   stated. Use the numeric value (5000, not "5000 dollars").
8. "payment_status" → one of: unpaid, partially_paid, paid, disputed.
9. "case_type" → a short classification (e.g. "contract_dispute"). This is the
   ONLY field you may infer from the topic of the message.

Previous assistant message (context only — do NOT copy facts from it):
\"\"\"{last_assistant_message}\"\"\"

Previous intake state (for reference — do NOT echo any of it):
{json.dumps(current_state, indent=2)}

Examples:

User: "Someone owes me money for freelance work and won't pay."
Output: {{"case_type": "contract_dispute", "payment_status": "unpaid"}}

User: "I'm in Texas."
Output: {{"state": "TX"}}

User: "I'm in California."
Output: {{"state": "CA"}}

User: "I have a written contract, I did $5000 worth of work."
Output: {{"written_contract_exists": true, "disputed_amount": 5000}}

User: "Yes."
Output: {{}}

Respond with ONLY a JSON object. No markdown. No commentary.
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"User Message: {user_query}"},
    ]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(
        **inputs, max_new_tokens=140,
        do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
    raw = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        clean = re.sub(r"```(?:json)?|```", "", raw).strip()
        delta_raw = json.loads(clean) if clean else {}
    except json.JSONDecodeError:
        logger.warning(f"Delta JSON parse failed; raw={raw!r}")
        return current_state

    delta = _validate_delta(delta_raw, state_indices)
    logger.info(f"Delta raw: {delta_raw}")
    logger.info(f"Delta validated: {delta}")

    new_state = dict(current_state)
    new_state.update(delta)
    return new_state

In [20]:
_VALID_INTENTS = {"new_legal_question", "clarification_answer", "off_topic", "chit_chat"}


def classify_input_intent(user_input, last_assistant_message, last_query):
    """Return one of: new_legal_question, clarification_answer, off_topic, chit_chat."""
    system_prompt = """You classify a user's message in an ongoing legal triage chat.

Return ONLY a JSON object: {"intent": "<one of: new_legal_question, clarification_answer, off_topic, chit_chat>"}

Definitions:
- new_legal_question: user asks a new legal question or describes a new legal
  situation that is not simply answering a clarification question.
- clarification_answer: user is directly answering one of the clarification
  questions in the Previous Assistant Message, OR is adding a factual detail
  about the current case.
- off_topic: user asks for something unrelated to legal triage (e.g. writing
  an essay, coding help, recipes, general trivia).
- chit_chat: greetings, thanks, small talk, filler.

When the user's message is short, check whether it directly answers one of the
clarification questions in the Previous Assistant Message. If yes, label it
clarification_answer. Otherwise, if it does not relate to the current case or
legal topic, label it off_topic.

Respond with ONLY the JSON object."""

    user_block = (
        f"Previous Assistant Message:\n\"\"\"{last_assistant_message}\"\"\"\n\n"
        f"Last substantive legal query:\n\"\"\"{last_query or '(none yet)'}\"\"\"\n\n"
        f"User Message:\n\"\"\"{user_input}\"\"\""
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_block},
    ]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(
        **inputs, max_new_tokens=60,
        do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
    raw = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        clean = re.sub(r"```(?:json)?|```", "", raw).strip()
        parsed = json.loads(clean)
        intent = parsed.get("intent", "").strip().lower()
        if intent in _VALID_INTENTS:
            return intent
    except Exception:
        logger.warning(f"Intent parse failed; raw={raw!r}")

    # conservative fallback
    t = user_input.strip().lower()
    if any(t.startswith(w) for w in ("what", "how", "why", "when", "where", "who",
                                     "can i", "should i", "is it", "do i", "does")):
        return "new_legal_question"
    return "clarification_answer"


# Fallback regex extractors
_STATE_REGEX = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in _STATE_NAME_TO_CODE.keys()) + r")\b",
    re.IGNORECASE,
)
_AMOUNT_REGEX = re.compile(
    r"\$\s*([\d,]+(?:\.\d+)?)\s*([kmb])?|\b([\d,]+(?:\.\d+)?)\s*(dollars?|bucks?|usd|k|m|b)\b",
    re.IGNORECASE,
)


def fallback_extract_state(text, valid_codes):
    m = _STATE_REGEX.search(text)
    if m:
        return normalize_state(m.group(1), valid_codes)
    return None


def fallback_extract_amount(text):
    m = _AMOUNT_REGEX.search(text)
    if not m:
        return None
    if m.group(1):
        v = float(m.group(1).replace(",", ""))
        if m.group(2):
            v *= _MULTIPLIERS[m.group(2).lower()]
        return v
    if m.group(3):
        v = float(m.group(3).replace(",", ""))
        unit = (m.group(4) or "").lower()
        if unit in _MULTIPLIERS:
            v *= _MULTIPLIERS[unit]
        return v
    return None

In [21]:
def _rrf_fuse(vector_ranking, bm25_ranking, k=60):
    scores = {}
    for rank, idx in enumerate(vector_ranking):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    for rank, idx in enumerate(bm25_ranking):
        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


def _bm25_query_tokens(bm25_index, clean_query):
    raw = clean_query.lower().split()
    corpus = getattr(bm25_index, "corpus", None)
    if not corpus or not isinstance(corpus[0], list):
        return raw
    vocab = set(t for doc in corpus for t in doc)
    if not vocab:
        return raw
    filtered = [w for w in raw if w in vocab]
    return filtered if filtered else raw


_CASE_QUERY_TEMPLATES = {
    "contract_dispute": "breach of contract, suit on account, open account, recovery of unpaid debts, damages, and attorney's fees",
    "landlord_tenant": "landlord tenant law, eviction, lease disputes, and recovery of rent",
    "employment": "employment law, unpaid wages, wrongful termination, wage claims",
    "personal_injury": "personal injury, negligence, tort damages",
    "family": "family law, divorce, child custody, child support, spousal support",
    "consumer": "consumer protection, deceptive trade practices, warranty claims",
    "insurance": "insurance claims, coverage disputes, insurer obligations",
}

_CASE_SIGNALS = {
    "contract_dispute": (
        "breach of contract", "suit on account", "open account",
        "sworn account", "contract debt", "unpaid invoice",
        "attorney's fees", "attorney fees", "mechanic's lien",
        "mechanics lien", "recovery of debt", "recovery of attorney",
        "contract", "debt", "damages", "remedies",
    ),
    "landlord_tenant": ("landlord", "tenant", "evict", "lease", "premises"),
    "employment": ("wage", "employ", "worker", "labor"),
    "personal_injury": ("negligence", "personal injury", "tort"),
    "family": ("divorce", "custody", "child support", "marriage"),
    "consumer": ("consumer protection", "deceptive trade", "warranty"),
    "insurance": ("insurer", "insurance policy", "coverage"),
}

_CASE_ANTI_SIGNALS = {
    "contract_dispute": (
        "insurance", "insurer", "policy", "coverage", "benefit",
        "landlord", "tenant", "evict", "lease", "premises",
        "employ", "wage", "worker",
        "divorce", "marriage", "custody",
        "criminal", "felony", "misdemeanor", "penal",
        "traffic", "vehicle", "motor", "toll",
        "medical", "health", "hospital",
        "legal services", "contingent fee", "barratry",
        "natural resources", "mining", "oil and gas",
        "county", "municipal",
        "frivolous", "state agency", "preservation",
        "historical", "archives", "library",
    ),
    "landlord_tenant": (
        "insurance", "insurer", "policy", "coverage",
        "criminal", "felony", "misdemeanor",
        "family", "divorce", "custody",
        "natural resources", "mining", "transportation",
        "frivolous", "state agency",
    ),
    "employment": (
        "insurance", "insurer", "policy", "coverage",
        "landlord", "tenant", "evict",
        "criminal", "felony", "misdemeanor",
        "natural resources", "mining", "transportation",
        "frivolous", "state agency",
    ),
    "personal_injury": (
        "insurance", "insurer", "policy",
        "landlord", "tenant", "evict",
        "transportation", "natural resources", "mining",
        "frivolous", "state agency",
    ),
}

_CASE_CODE_ANTI = {
    "contract_dispute": (
        "insurance code", "transportation code", "natural resources code",
        "local government code", "health and safety code", "penal code",
        "tax code", "utilities code", "water code", "parks and wildlife code",
        "agriculture code", "alcoholic beverage code", "education code",
        "election code", "code of criminal procedure",
        "labor code", "human resources code", "family code",
        "government code", "property code",
    ),
    "landlord_tenant": (
        "insurance code", "transportation code", "natural resources code",
        "health and safety code", "penal code", "tax code", "utilities code",
        "water code", "agriculture code", "education code", "election code",
        "code of criminal procedure", "labor code",
    ),
    "employment": (
        "insurance code", "transportation code", "natural resources code",
        "local government code", "health and safety code", "penal code",
        "utilities code", "water code", "parks and wildlife code",
        "agriculture code", "alcoholic beverage code", "election code",
        "family code",
    ),
    "personal_injury": (
        "insurance code", "transportation code", "natural resources code",
        "local government code", "tax code", "utilities code", "water code",
        "agriculture code", "alcoholic beverage code", "election code",
        "family code",
    ),
}


def _soft_score(case_type, meta):
    """Soft scoring: positive signals add, anti-signals subtract."""
    ct = (case_type or "").lower()
    signals = _CASE_SIGNALS.get(ct, ())
    antis = _CASE_ANTI_SIGNALS.get(ct, ())
    code_antis = _CASE_CODE_ANTI.get(ct, ())

    header = " ".join([
        meta.get("citation") or "",
        meta.get("citation_short") or "",
        meta.get("section_title") or "",
    ]).lower()
    citation = (meta.get("citation") or "").lower()
    body = (meta.get("full_text") or "").lower()

    score = 0.0
    for s in signals:
        if s in header:
            score += 2.0
        elif s in body:
            score += 0.5
    for a in antis:
        if a in header:
            score -= 3.0
        elif a in body:
            score -= 0.3
    for a in code_antis:
        if a in citation:
            score -= 5.0
    return score


_MIN_COS = 0.45            # relaxed a bit
_MIN_SOFT_SCORE = -1.0     # allow mild anti-signals
_MIN_BM25_ONLY_RANK = 5    # BM25-only candidates must be in the BM25 top 5


def _retrieve_statutes(state_code, clean_query, case_type, top_k=3, pool=40):
    if not state_code or state_code not in state_indices:
        return []

    domain_phrase = _CASE_QUERY_TEMPLATES.get((case_type or "").lower(), "legal remedies")
    statute_query = f"{state_code.upper()} statutes on {domain_phrase}. {clean_query}"

    q_vec = embedding_model.encode(
        ["Represent this sentence for searching relevant passages: " + statute_query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cpu",
    ).astype("float32")

    cos_scores, vec_idx = state_indices[state_code].search(q_vec, pool)
    vec_ranking = [int(i) for i in vec_idx[0]]
    vec_sims = {int(i): float(s) for i, s in zip(vec_idx[0], cos_scores[0])}

    bm25 = state_bm25[state_code]
    q_tokens = _bm25_query_tokens(bm25, clean_query)
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_top = list(np.argsort(bm25_scores)[::-1][:pool])
    bm25_ranking = [int(i) for i in bm25_top]
    bm25_rank_map = {int(i): r for r, i in enumerate(bm25_ranking)}

    # UNION of vector and BM25 candidates -> RRF fuse
    fused = _rrf_fuse(vec_ranking, bm25_ranking)

    scored = []
    for idx, rrf in fused:
        cos = vec_sims.get(idx)
        bm25_rank = bm25_rank_map.get(idx)

        # keep BM25-only candidates only if they were high-ranked
        if cos is None and (bm25_rank is None or bm25_rank > _MIN_BM25_ONLY_RANK):
            continue
        if cos is not None and cos < _MIN_COS:
            continue

        doc = state_docs_map[state_code][idx]
        soft = _soft_score(case_type, doc["metadata"])
        if soft < _MIN_SOFT_SCORE:
            continue

        scored.append((doc, rrf, cos if cos is not None else 0.0, soft))

    # sort by soft score, then RRF
    scored.sort(key=lambda x: (-x[3], -x[1]))
    return scored[:top_k]


def _qa_state_ok(doc, state_code):
    doc_state = (doc.get("metadata", {}).get("state") or "").lower().strip()
    if not doc_state:
        return True
    if not state_code:
        return False
    return doc_state == state_code

In [22]:
def _build_citation_rule(state_code, n_candidates, corpus_loaded):
    if not state_code:
        return (
            "2. The user has NOT provided a state. You MUST NOT cite any statute, "
            "and you MUST NOT guess one. Instead, ask the user which US state "
            "they are in (or 'federal') to proceed."
        )
    if not corpus_loaded:
        return (
            f"2. No statute corpus is loaded for '{state_code.upper()}'. You MUST "
            "NOT cite any statute. Say plainly that the statute corpus for this "
            "jurisdiction is unavailable."
        )
    if n_candidates == 0:
        return (
            f"2. The statute corpus for '{state_code.upper()}' was searched but "
            "no candidates passed the relevance filters. You MUST NOT cite any "
            "statute. Say plainly: \"No relevant statute was found in the "
            f"{state_code.upper()} corpus for this situation.\" Do NOT tell the "
            "user to identify their state — the state is already known."
        )
    return (
        f"2. The 'Retrieved Statutes' block contains {n_candidates} candidate "
        "statute(s). Read each carefully.\n"
        "   (a) IF at least one candidate genuinely governs the user's situation "
        "— i.e. it actually addresses the relevant legal theory, remedy, or "
        "procedure — you MUST reference at least one of them by name. Copy the "
        "citation EXACTLY as it appears in the 'Citation:' line.\n"
        "   (b) IF NONE of the candidates match the subject matter, you MUST NOT "
        "cite any of them. State plainly: \"The retrieved statutes do not "
        "directly address this situation, so I cannot cite a specific statute.\"\n"
        "   (c) Do NOT stretch, rationalize, or analogize an unrelated statute "
        "to make it fit. It is better to cite no statute than to cite the wrong one."
    )


def get_legal_triage(clean_query, current_state, top_k_qa=2, top_k_statutes=3):
    state_code = (current_state.get("state") or "").lower().strip()
    case_type = current_state.get("case_type") or "Unknown"

    # --- Q&A retrieval ---
    search_string = f"Area of law: {case_type}. Situation: {clean_query}"
    qa_vector = embedding_model.encode(
        ["Represent this sentence for searching relevant passages: " + search_string],
        convert_to_numpy=True, normalize_embeddings=True, device="cpu",
    ).astype("float32")

    qa_pool = max(top_k_qa * 6, 12)
    _, qa_indices = qa_index.search(qa_vector, qa_pool)
    qa_hits = []
    for i in qa_indices[0]:
        doc = rag_documents[int(i)]
        if _qa_state_ok(doc, state_code):
            qa_hits.append(doc)
        if len(qa_hits) >= top_k_qa:
            break

    qa_context = "\n\n".join(
        f"Practical Guidance:\n{d['metadata']['legal_guidance']}" for d in qa_hits
    ) or "(no guidance retrieved)"

    # --- Statute retrieval ---
    statute_context = ""
    n_candidates = 0
    corpus_loaded = bool(state_code and state_code in state_indices)

    if corpus_loaded:
        hits = _retrieve_statutes(
            state_code, clean_query,
            current_state.get("case_type") or "",
            top_k=top_k_statutes,
        )
        n_candidates = len(hits)
        if n_candidates == 0:
            statute_context = (
                f"(The {state_code.upper()} corpus was searched; no statute "
                "passed the relevance filters for this query.)\n"
            )
        else:
            for doc, rrf, cos, soft in hits:
                meta = doc["metadata"]
                statute_context += (
                    f"\n---\n"
                    f"Citation: {meta['citation']}\n"
                    f"State: {meta.get('state')}\n"
                    f"Title: {meta.get('section_title', '')}\n"
                    f"Text: {meta['full_text'][:2000]}\n"
                )
    else:
        statute_context = "(No statute corpus available for this jurisdiction.)\n"

    # --- Known facts guard ---
    known_lines = []
    if current_state.get("state"):
        known_lines.append(f"- State is already known: {current_state['state'].upper()} — DO NOT ask which state.")
    if current_state.get("case_type"):
        known_lines.append(f"- Case type is already known: {current_state['case_type']} — DO NOT ask what kind of case this is.")
    if current_state.get("disputed_amount") is not None:
        known_lines.append(f"- Disputed amount is already known: ${current_state['disputed_amount']:.2f} — DO NOT ask for the amount.")
    if current_state.get("written_contract_exists") is not None:
        known_lines.append(f"- Written contract status is already known: {current_state['written_contract_exists']} — DO NOT ask whether there is a written contract.")
    if current_state.get("payment_status"):
        known_lines.append(f"- Payment status is already known: {current_state['payment_status']} — DO NOT ask about payment status.")
    known_facts_guard = "\n".join(known_lines) or "- (no facts known yet)"

    cite_rule = _build_citation_rule(state_code, n_candidates, corpus_loaded)

    system_prompt = f"""You are a strict legal triage assistant.

STRICT GROUNDING RULES (violating any of these is a failure):
1. Rely ONLY on the two context blocks below.
{cite_rule}
3. NEVER copy a statute citation out of the "Retrieved Q&A Guidance" block.
   Q&A guidance is generic practice advice; it is NOT a citation source.
4. When referencing a statute, write only the citation in prose. NEVER reproduce
   the raw "Citation: ... / State: ... / Title: ... / Text: ..." block verbatim.
5. Do not assign gender to any party unless the user explicitly stated it.
6. Do NOT tell the user to send anything "on attorney letterhead" unless the
   user has stated they are an attorney.
7. Only mention a mechanics lien if the user's description involves construction
   or an improvement to real property.
8. Do not reference case numbers, docket numbers, or party names unless the user
   mentioned them.
9. If the user asks to conceal assets, evade taxes, destroy evidence, or commit
   illegal acts, refuse that request and outline lawful alternatives only.

FACTS YOU MUST NOT ASK ABOUT AGAIN (they are already known):
{known_facts_guard}

CLARIFICATION QUESTION RULES:
- Ask ONLY about fields that are still null/unknown.
- NEVER ask "Which US state are you in?" if the state is already known.
- If nothing is unknown, write exactly: (none — all required facts are known)
- Each question must be on its own line, starting with "- ".

Known Facts (Null = unknown — do NOT guess a value for any null field):
{json.dumps(current_state, indent=2)}

Retrieved Q&A Guidance:
{qa_context}

Retrieved Statutes:
{statute_context}

OUTPUT FORMAT — use these EXACT Markdown headers, in this exact order:

### Initial Assessment
<1-2 short paragraphs>

### Action Steps You Can Take Now
<numbered list>

### Clarification Questions
<bullet list, one per line, each starting with "- ", OR the literal text
"(none — all required facts are known)">

Speak plainly. Do not add any prose before the first "###" header or after the
last section.
"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"My Situation:\n{clean_query}"},
    ]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(llm_model.device)
    outputs = llm_model.generate(
        **inputs, max_new_tokens=600,
        do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [25]:
intake_state = {
    "case_type": None,
    "state": None,
    "disputed_amount": None,
    "written_contract_exists": None,
    "payment_status": None,
}
last_substantive_query = None
last_assistant_message = "This is the start of the conversation. Ask the user how you can help."


def _off_topic_response(intent):
    if intent == "chit_chat":
        return ("Hello! I'm a legal triage assistant. Tell me about a legal "
                "situation — for example, an unpaid invoice, a landlord "
                "dispute, or a contract problem — and I'll help you triage it.")
    return ("I'm a legal triage assistant, so I can only help with legal "
            "questions. Could you describe a legal issue you're facing?")


def _enrich_query(prev_query, new_input, last_asst):
    """Merge a new fact into the previous query."""
    return translate_query(
        raw_user_input=new_input,
        last_assistant_message=last_asst,
        previous_query=prev_query,
    )


def _apply_fallbacks(user_input, current_state):
    """Regex fallbacks for state and amount if the LLM missed them."""
    state = current_state.get("state")
    if state is None:
        s = fallback_extract_state(user_input, state_indices)
        if s:
            logger.info(f"Fallback state extracted: {s}")
            current_state = {**current_state, "state": s}

    amt = current_state.get("disputed_amount")
    if amt is None:
        a = fallback_extract_amount(user_input)
        if a is not None:
            logger.info(f"Fallback amount extracted: {a}")
            current_state = {**current_state, "disputed_amount": a}

    return current_state

In [26]:
def start_legal_chat():
    global intake_state, last_substantive_query, last_assistant_message

    print("=" * 60)
    print("  LEGAL ASSISTANT")
    print("Type 'quit' or 'exit' to end.")
    print("=" * 60)

    while True:
        user_input = input("\nYou: ")
        if user_input.strip().lower() in ("quit", "exit"):
            break
        if not user_input.strip():
            continue

        # 1. Classify intent
        intent = classify_input_intent(
            user_input, last_assistant_message, last_substantive_query
        )
        logger.info(f"Intent: {intent}")

        # 2. Handle off-topic and chit-chat immediately
        if intent in ("off_topic", "chit_chat"):
            reply = _off_topic_response(intent)
            print("\nAssistant:")
            print(reply)
            print("-" * 60)
            last_assistant_message = reply
            continue

        # 3. Build the search query
        if intent == "new_legal_question":
            clean_query = translate_query(user_input, last_assistant_message)
            last_substantive_query = clean_query
            print(f"[Search: {clean_query}]")
        else:  # clarification_answer
            if last_substantive_query:
                clean_query = _enrich_query(
                    last_substantive_query, user_input, last_assistant_message
                )
                last_substantive_query = clean_query
                print(f"[Search (updated): {clean_query}]")
            else:
                clean_query = translate_query(user_input, last_assistant_message)
                last_substantive_query = clean_query
                print(f"[Search: {clean_query}]")

        # 4. Update intake state (LLM + fallback extractors)
        before = dict(intake_state)
        intake_state = update_intake_state(
            user_input, intake_state, state_indices, last_assistant_message
        )
        intake_state = _apply_fallbacks(user_input, intake_state)

        changed = {k: v for k, v in intake_state.items() if before.get(k) != v}
        if changed:
            logger.info(f"State updated: {changed}")
            print(f"[Facts updated: {changed}]")

        # 5. Get legal triage response
        response = get_legal_triage(clean_query, intake_state)
        last_assistant_message = response

        print("\nAssistant:")
        print(response)
        print("-" * 60)

In [ ]:
start_legal_chat()

  LEGAL ASSISTANT
Type 'quit' or 'exit' to end.



You:  what is contract?


2026-09-22 14:37:59,198 [INFO] Intent: new_legal_question


[Search: Query: Define "contract" in legal terms.]


2026-09-22 14:38:01,287 [INFO] Delta raw: {}
2026-09-22 14:38:01,288 [INFO] Delta validated: {}


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Assistant:
### Initial Assessment
In Arizona, an implied contract can be formed without an express agreement between the parties, based on the conduct and circumstances indicating that both parties intended to form a contract. To file a claim against a deceased party's estate, the claimant must establish that an implied contract existed, which requires proving the elements of a valid contract: offer, acceptance, consideration, and mutual intent to be bound.

The nature of the relationship between the claimant and the deceased, as well as the benefits or consideration exchanged, are crucial in determining the existence of an implied contract. If the claimant and the deceased had a business relationship, provided services or goods to the deceased, or the deceased benefited from the claimant's actions, these could be considered as consideration for an implied contract.

It is advisable for the claimant to consult with an attorney to assess the strength of their claim against the deceased


You:  if someone steal a car does that comes in the contract?


2026-09-22 15:04:14,912 [INFO] Intent: off_topic



Assistant:
I'm a legal triage assistant, so I can only help with legal questions. Could you describe a legal issue you're facing?
------------------------------------------------------------
